In [ ]:
import os
from dotenv import load_dotenv
import duckdb

load_dotenv(override=True)

con = duckdb.connect()

In [2]:
path = "s3://mateom/graal/embeddings/NAF2025/2026-03-16_gemma_synth.parquet"

In [26]:
code_len_query = f"""
    SELECT COUNT(DISTINCT(code))
    FROM read_parquet('{path}')
    """

n_codes = con.execute(code_len_query).fetchone()[0]

In [27]:
n_codes

747

In [44]:
random_state = 42

In [54]:
counts = con.execute(f"""
    SELECT code, COUNT(*) AS cnt
    FROM read_parquet('{path}')
    GROUP BY code
    ORDER BY code
""").df()

In [56]:
import numpy as np

rng = np.random.default_rng(seed=random_state)

counts["k"] = counts["cnt"].apply(
    lambda n: rng.integers(1, n + 1)
)

In [57]:
counts["offset"] = counts["cnt"].cumsum() - counts["cnt"]
counts["row_id"] = counts["offset"] + counts["k"]

In [74]:
selected_ids = counts[["row_id"]]
con.register("selected_ids", selected_ids)

In [128]:
selected_ids = counts["row_id"].to_numpy()
remaining_ids = np.delete(np.arange(1,counts["cnt"].sum()+1),selected_ids-1)
random_ids = rng.choice(remaining_ids, 300)
final_ids = np.concatenate([selected_ids, random_ids])
con.register("selected_ids", {"row_id":final_ids})

In [130]:
query = f"""
SELECT *
FROM selected_ids
"""

res = con.execute(query).df()

In [132]:
query = f"""
SELECT t.code, t.embedding
FROM (
    SELECT *,
           ROW_NUMBER() OVER (ORDER BY code) AS row_id
    FROM read_parquet('{path}')
) t
JOIN selected_ids s
USING (row_id)
"""

res = con.execute(query).df()

In [133]:
res.head()

,code,embedding
0,96.30Y,"[0.039255790412425995, 0.01053204108029604, -0..."
1,96.40Y,"[0.023108962923288345, 0.015642991289496422, 0..."
2,96.40Y,"[0.029340434819459915, 0.02590949647128582, 0...."
3,96.91Y,"[0.015917960554361343, 0.031123176217079163, 0..."
4,96.99G,"[0.013327492401003838, 0.010817381553351879, 0..."


In [135]:
code_list = res["code"].to_list()

In [140]:
from collections import Counter

code_counts = Counter(code_list)

In [141]:
code_counts["01.11Y"]

6

In [139]:
code_row_count = con.execute(f"""
    SELECT code, COUNT(*) AS n_rows
    FROM read_parquet('{path}')
    GROUP BY code
    ORDER BY code
    """).df()

In [149]:
code_row_count

,code,n_rows
0,01.11Y,400
1,01.12Y,50
2,01.13Y,350
3,01.14Y,50
4,01.15Y,50
...,...,...
742,96.99H,150
743,97.00Y,50
744,98.10Y,50
745,98.20Y,50


In [153]:
code_row_count.apply(
    lambda x: rng.integers(1, x["n_rows"] + 1, size=code_counts[x["code"]]), axis=1
).explode()

0      374
0      148
0       52
0      168
0       52
      ... 
742     57
743     27
744     34
745     42
746     23
Length: 1047, dtype: object